# IEEE-CIS Fraud Detection - Model Inference

This notebook:

1. Loads the **best model's Pipeline from the MLflow Model Registry** (a single
   sklearn `Pipeline`, so we don't have to repeat preprocessing here).
2. Reads the RAW `test_transaction.csv` and `test_identity.csv` and merges them.
3. Calls `pipeline.predict_proba(raw_X)`.
4. Writes a Kaggle-ready `submission.csv` (`TransactionID,isFraud`).

To switch which architecture is used, change `REGISTERED_NAME` below to whichever
`IEEE_Fraud_<Model>` has the highest CV AUC after running all experiment notebooks.


## 1. Load best Pipeline from MLflow Model Registry

In [ ]:
import os, gc, pandas as pd, numpy as np
import mlflow, mlflow.sklearn, dagshub
import warnings; warnings.filterwarnings('ignore')

REPO_OWNER = "rkvit23"
REPO_NAME  = "ML-HW2"
dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

# Pick the model to use. After all experiments are done you can change this
# to whichever architecture has the best CV AUC.
REGISTERED_NAME = "IEEE_Fraud_XGBoost"   # <-- change to the winning architecture
MODEL_URI       = f"models:/{REGISTERED_NAME}/latest"

print(f"Loading {MODEL_URI} from MLflow ...")
pipeline = mlflow.sklearn.load_model(MODEL_URI)
print("Loaded:", type(pipeline))


## 2. Predict on raw test set

In [ ]:
DATA_DIR = "data"   # change to "/kaggle/input/ieee-fraud-detection" on Kaggle

print("Loading raw test data...")
test_tx = pd.read_csv(os.path.join(DATA_DIR, "test_transaction.csv"))
test_id = pd.read_csv(os.path.join(DATA_DIR, "test_identity.csv"))
test_id.columns = [c.replace('-', '_') for c in test_id.columns]
test = test_tx.merge(test_id, on="TransactionID", how="left")
del test_tx, test_id; gc.collect()

test_ids = test["TransactionID"].copy()
X_raw    = test.drop(columns=["TransactionID"])
print(f"Raw test shape: {X_raw.shape}")

# The pipeline includes ALL preprocessing -> preds in one call
print("Predicting...")
preds = pipeline.predict_proba(X_raw)[:, 1]
print(f"Done. mean P(fraud) = {preds.mean():.5f}, min={preds.min():.5f}, max={preds.max():.5f}")


## 3. Generate `submission.csv`

In [ ]:
submission = pd.DataFrame({"TransactionID": test_ids, "isFraud": preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved:", submission.shape)
submission.head()


## 4. Sanity-check predictions

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,5))
ax.hist(preds, bins=80, color='steelblue', edgecolor='black')
ax.set_xlabel('Predicted P(isFraud)'); ax.set_ylabel('count')
ax.set_title('Predicted fraud-probability distribution (test set)')
plt.tight_layout(); plt.show()

print("Top-10 highest-risk transactions:")
print(submission.sort_values('isFraud', ascending=False).head(10).to_string(index=False))
